# RollOCT Quickstart

This notebook demonstrates the basic usage of **RollOCT** — a rolling lookahead optimal classification tree.

We'll cover:
1. Loading data
2. Training a model
3. Making predictions
4. Evaluating accuracy
5. Inspecting the fitted tree

## 1. Setup

Make sure you've installed the dependencies:
```bash
pip install -r requirements.txt
```

In [ ]:
import pandas as pd
import numpy as np
from rollo_oct import RollingOCT

## 2. Load Data

We'll use the bundled Wine dataset (binarized version of the [UCI Wine Dataset](https://archive.ics.uci.edu/dataset/109/wine)).

- **3 classes** (wine varieties)
- **130 binary features** (one-hot encoded from the original 13 continuous features)
- Target column: `y`

In [ ]:
train = pd.read_csv("../rollo_oct/data/train.csv")
test = pd.read_csv("../rollo_oct/data/test.csv")

print(f"Training samples: {len(train)}")
print(f"Test samples:     {len(test)}")
print(f"Features:         {train.shape[1] - 1}")
print(f"Classes:          {sorted(train['y'].unique())}")
train.head()

In [ ]:
# Separate features and target
X_train = train.drop("y", axis=1)
y_train = train["y"]

X_test = test.drop("y", axis=1)
y_test = test["y"]

print(f"Class distribution (train): {y_train.value_counts().to_dict()}")
print(f"Class distribution (test):  {y_test.value_counts().to_dict()}")

## 3. Train a Model

Create a `RollingOCT` classifier and call `fit()`. The API follows the familiar sklearn pattern.

Key parameters:
- `depth`: Maximum tree depth (>= 2)
- `criterion`: `"gini"` or `"misclassification"`
- `solver`: `"highs"` (default, open-source), `"gurobi"`, or `"cbc"`

In [ ]:
# Train a depth-2 tree (base case — single OCT-2 solve)
model_d2 = RollingOCT(depth=2, criterion="gini", solver="highs")
model_d2.fit(X_train, y_train)

print(f"Depth-2 train accuracy: {model_d2.score(X_train, y_train):.3f}")
print(f"Depth-2 test accuracy:  {model_d2.score(X_test, y_test):.3f}")

## 4. Deeper Trees with Rolling Optimization

Setting `depth > 2` triggers the rolling subtree algorithm:
the initial depth-2 tree is expanded by solving additional depth-2 subproblems
at misclassified leaves.

In [ ]:
# Train a depth-3 tree (one round of rolling expansion)
model_d3 = RollingOCT(depth=3, criterion="gini", solver="highs")
model_d3.fit(X_train, y_train)

print(f"Depth-3 train accuracy: {model_d3.score(X_train, y_train):.3f}")
print(f"Depth-3 test accuracy:  {model_d3.score(X_test, y_test):.3f}")

## 5. Making Predictions

In [ ]:
# Predict on test data
predictions = model_d3.predict(X_test)

print(f"Predictions shape: {predictions.shape}")
print(f"First 10 predictions: {predictions[:10]}")
print(f"First 10 actual:      {y_test.values[:10]}")

## 6. Inspecting Results

After fitting, the model exposes useful attributes for analysis.

In [ ]:
# Per-depth results (accuracy and timing at each depth level)
for depth, result in sorted(model_d3.depth_results_.items()):
    print(
        f"Depth {depth}: "
        f"train_acc={result.training_accuracy:.3f}, "
        f"test_acc={result.test_accuracy:.3f}, "
        f"time={result.elapsed_time:.2f}s"
    )

In [ ]:
# Inspect the tree structure
tree = model_d3.tree_

print(f"Tree depth: {tree.depth}")
print(f"Branch nodes: {sorted(tree.branch_nodes.keys())}")
print(f"Leaf nodes:   {sorted(tree.leaf_nodes.keys())}")
print()

# Show which feature each branch node splits on
for nid, node in sorted(tree.branch_nodes.items()):
    if node.feature_index is not None:
        print(f"Node {nid}: splits on feature {node.feature_index}")

print()

# Show leaf predictions
for lid, leaf in sorted(tree.leaf_nodes.items()):
    if leaf.predicted_class is not None:
        status = " (pruned)" if leaf.is_pruned else ""
        print(f"Leaf {lid}: predicts class {leaf.predicted_class}{status}")

## 7. Using Numpy Arrays

RollingOCT accepts both DataFrames and numpy arrays.

In [ ]:
# Works with plain numpy arrays too
model_np = RollingOCT(depth=2, solver="highs")
model_np.fit(X_train.values, y_train.values)
print(f"Accuracy (numpy input): {model_np.score(X_test.values, y_test.values):.3f}")

## Summary

| Step | Code |
|------|------|
| Create model | `model = RollingOCT(depth=3, solver="highs")` |
| Train | `model.fit(X_train, y_train)` |
| Predict | `preds = model.predict(X_test)` |
| Evaluate | `acc = model.score(X_test, y_test)` |
| Inspect tree | `model.tree_.branch_nodes`, `model.tree_.leaf_nodes` |
| Per-depth stats | `model.depth_results_` |

For more advanced usage (comparing solvers, criteria, and depths), see **02_advanced.ipynb**.